# Do Chatbot ao Agente: Construindo Sistemas de IA com Ferramentas

Neste notebook você roda, de ponta a ponta, um agente que **raciocina em passos** usando ferramentas:

1. **Modelo** (Groq, via LangChain), que só *pede* chamadas de ferramenta: quem executa é o nosso código.
2. **Memória de trabalho**: a API é *stateless*, então o histórico completo é reenviado a cada passo.
3. **Registry de ferramentas**: `update_plan` (o agente registra o raciocínio) e `finish` (entrega a resposta e encerra).
4. **Loop**: chama o modelo, executa as ferramentas pedidas e repete até `finish` (ou até `max_steps`).
5. **Tracing**: tokens, latência e plano de cada passo.

**Antes de começar:** no painel **Secrets** (ícone de chave 🔑 à esquerda), crie o secret `GROQ_API_KEY` com a **sua** chave
do [console do Groq](https://console.groq.com/keys) e **ative "Acesso ao notebook"** para este notebook. Opcional:
`GROQ_MODEL` para trocar o modelo. Nunca cole a chave numa célula.

## 1. Preparar o ambiente

Clona o repositório (ou atualiza, se já existir), instala as dependências e coloca o pacote no `sys.path`.
Localmente (VS Code/Jupyter com o kernel do `.venv`), usa o próprio repositório, sem clonar nem instalar.

> Se você atualizar o repositório **depois** de importar o pacote, o Python continua com a versão antiga na memória:
> reinicie o ambiente de execução (*Ambiente de execução → Reiniciar sessão*) e rode as células de novo
> (ou use `importlib.reload()`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pedroop32/workshop-agentes.git"
REPO_DIR = "/content/workshop-agentes"
IN_COLAB = "google.colab" in sys.modules


def sh(*cmd: str) -> None:
    """Roda um comando; mostra a saída só se der erro."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout, result.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", REPO_URL, REPO_DIR)
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")
else:
    # Local: a pasta do repositório é a atual ou uma acima dela.
    local = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_agentes").is_dir()), None)
    assert local, "Abra o notebook de dentro da pasta do repositório (onde está workshop_agentes/)."
    REPO_DIR = str(local)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Ambiente pronto ({'Colab' if IN_COLAB else 'local'}): {REPO_DIR}")

## 2. Configuração e teste de conexão

Lê a chave e o modelo dos Secrets (`get_secret`: Colab primeiro, depois variável de ambiente/`.env`) e faz **uma** chamada
simples para confirmar que tudo funciona. Em caso de falha, mostramos só o **tipo** do erro, nunca a mensagem
(ela poderia expor dados). A chamada passa por `invoke_with_retry`, o mesmo retry com limite do agente: um 429 por
minuto é retentado; se ainda falhar, mostramos qual limite estourou (RPM, TPM, RPD ou TPD) e quanto esperar.
Se o modelo não for encontrado, listamos os modelos ativos para a sua chave.

Também imprimimos `usage_metadata` (inclui tokens de raciocínio e de cache, quando o modelo informa) e `finish_reason`,
os dados que o tracing usa.

In [ ]:
import groq
from langchain_core.messages import HumanMessage

from workshop_agentes.agent.config import DEFAULT_MODEL, AgentConfig, get_secret
from workshop_agentes.llm import (
    DailyQuotaError,
    build_llm,
    invoke_with_retry,
    list_models,
    rate_limit_window,
    text_of,
)

GROQ_API_KEY = get_secret("GROQ_API_KEY")
assert GROQ_API_KEY, "Crie o secret GROQ_API_KEY (ícone de chave) e ative o acesso do notebook."
MODEL = get_secret("GROQ_MODEL", DEFAULT_MODEL)
print(f"Modelo: {MODEL}")

config = AgentConfig()  # max_steps=8, temperatura padrão do ChatGroq, verbose=True
llm = build_llm(MODEL, GROQ_API_KEY, temperature=config.temperature)

try:
    reply, attempts = invoke_with_retry(llm, [HumanMessage(content="Responda apenas: conexão funcionando!")])
    print(f"SUCESSO: {text_of(reply.content)} (tentativas: {attempts})")
    print(f"usage_metadata: {reply.usage_metadata}")
    print(f"finish_reason: {reply.response_metadata.get('finish_reason')}")
except Exception as exc:
    print(f"FALHA: {type(exc).__name__}")
    if isinstance(exc, DailyQuotaError):
        print(exc)  # mensagem nossa, sem dados da conta
    elif isinstance(exc, groq.RateLimitError):
        wait = exc.response.headers.get("retry-after")
        print(f"Limite atingido: {rate_limit_window(exc)}; a API pede esperar {wait} s antes de tentar de novo.")
    elif isinstance(exc, groq.NotFoundError):
        try:
            print("Modelos ativos (defina o secret GROQ_MODEL com um que aceite ferramentas):")
            for name in list_models(GROQ_API_KEY):
                print(" -", name)
        except Exception as list_exc:
            print(f"Não foi possível listar os modelos: {type(list_exc).__name__}")

## 3. Montar o agente

Cada peça é criada separadamente e **injetada** no agente (injeção de dependências): é fácil trocar o modelo,
as ferramentas, a memória ou o tracer sem mexer no loop.

- `update_plan`: o modelo registra o que descobriu, o plano e o próximo passo; o campo `external` aparece ao vivo para você.
- **Resposta final em texto:** quando o modelo responde sem pedir ferramenta, o loop termina (`no_tool_call`), como no
  padrão da documentação do Groq.
- `finish` (opcional): uma ferramenta **terminal**. Para usá-la, `registry.register(finish, terminal=True)`; quando ela
  dá certo, o loop termina com a resposta.

In [ ]:
from workshop_agentes.agent.loop import Agent
from workshop_agentes.agent.memory import WorkingMemory
from workshop_agentes.tools.core import update_plan
from workshop_agentes.tools.registry import ToolRegistry
from workshop_agentes.tracing.tracer import Tracer

registry = ToolRegistry()
registry.register(update_plan)  # a resposta final vem em texto; finish é opcional (veja acima)

agent = Agent(
    llm=llm,
    registry=registry,
    memory=WorkingMemory(config.system_prompt),
    tracer=Tracer(),
    config=config,
)
print("Ferramentas:", [tool.name for tool in registry.tools])

## 4. Executar o agente

As linhas com `→` são o campo `external` de cada `update_plan`: o que o agente **vai fazer** em seguida.
No fim aparecem a resposta e o **motivo de término**:

| motivo | significado |
|---|---|
| `no_tool_call` | o modelo respondeu só com texto: é a resposta final |
| `length` | a resposta em texto foi cortada pelo limite de tokens (está incompleta) |
| `finish` | uma ferramenta terminal (ex.: `finish`, se registrada) deu certo |
| `max_steps` | atingiu o limite de passos sem resposta final |
| `error` | falha na chamada ao modelo (o tipo do erro aparece em `error`) |

O `finish_reason` é o motivo informado pelo próprio Groq na última resposta (`stop`, `tool_calls`, `length`).

Pode rodar esta célula várias vezes: cada `run()` começa com memória e tracing novos.
Lembre que **cada passo é uma chamada à API** e conta para os limites do plano gratuito.

In [ ]:
TASK = "Compare lista encadeada e array dinâmico para implementar uma fila e conclua qual usar."

result = agent.run(TASK)

print()
print(f"Motivo de término: {result.reason} | finish_reason do Groq: {result.finish_reason} | passos: {result.steps}")
if result.error:
    print(f"Erro: {result.error}")
print(f"Resposta final:\n{result.answer}")

## 5. Relatório de tracing

Tokens e latência **por passo**, **por função** e no **total**, com o `finish_reason` e o plano registrado em cada passo.
Os totais somam só as chamadas (folhas da árvore de spans), para não contar nada em dobro.

In [ ]:
from workshop_agentes.tracing.report import format_report

print(format_report(result.spans))
print()
print("Invariante da memória (toda chamada tem seu resultado logo depois):", agent.memory.check_invariant())

## Extras: o que olhar enquanto testa

Células opcionais para enxergar o que aconteceu no último `agent.run()` (seção 4). Só a E3 e a E4 chamam a API.

### E1. Memória da execução

Tudo o que o modelo recebeu na última chamada: a memória é a lista de mensagens do `run()`, reenviada inteira a cada
passo. Ela vale **só dentro de um run**: o próximo `agent.run()` começa de novo, só com o system prompt.

In [ ]:
import json

from langchain_core.messages import AIMessage, ToolMessage


def resumo(texto, limite: int = 160) -> str:
    """Texto numa linha só, cortado em `limite` caracteres."""
    texto = " ".join(str(texto).split())
    return texto if len(texto) <= limite else texto[: limite - 1] + "…"


def mostrar_memoria(res) -> None:
    """Uma linha por mensagem: quem falou e o quê (chamadas de ferramenta com os argumentos)."""
    for i, msg in enumerate(res.messages):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                print(f"[{i}] AI pede {call['name']}({resumo(json.dumps(call['args'], ensure_ascii=False))})")
        elif isinstance(msg, ToolMessage):
            print(f"[{i}] Tool {msg.name} → {resumo(msg.content)}")
        else:
            print(f"[{i}] {type(msg).__name__.removesuffix('Message')}: {resumo(text_of(msg.content))}")
    print(f"{len(res.messages)} mensagens")


mostrar_memoria(result)
print("Invariante (toda chamada tem seu resultado logo depois):", agent.memory.check_invariant())

### E2. Métricas do Groq por chamada

Para cada chamada ao modelo: tokens (inclusive de raciocínio e de cache, quando o modelo informa), `finish_reason`,
a latência medida por nós (span `llm_call`: inclui rede e esperas de retry) e o tempo medido pelo Groq no servidor e na
fila, e as tentativas que falharam e foram retentadas (tipo e limite, ex.: `RateLimitError OTPM`). Raciocínio e cache
só aparecem quando o modelo os reporta (o qwen, no padrão, não raciocina; o cache de prompt do Groq só existe nos
modelos gpt-oss). O que **não** dá para medir aqui: a quota restante (veja E3), o tempo até o primeiro token (não usamos streaming)
e o custo isolado de cada passo (o `input_tokens` inclui todo o histórico reenviado).

In [ ]:
def mostrar_metricas(res) -> None:
    """Tokens e tempos de cada chamada ao modelo do run."""
    spans = [s for s in res.spans if s.kind == "llm_call"]
    replies = [m for m in res.messages if isinstance(m, AIMessage)]
    for n, (span, reply) in enumerate(zip(spans, replies), start=1):
        usage = reply.usage_metadata or {}
        server = reply.response_metadata.get("token_usage") or {}  # o usage do Groq como veio
        # Raciocínio e cache só aparecem quando o Groq manda o campo (lidos do usage bruto).
        reasoning = (server.get("completion_tokens_details") or {}).get("reasoning_tokens")
        cached = (server.get("prompt_tokens_details") or {}).get("cached_tokens")
        extra = [f"{label}={value}" for label, value in (("raciocínio", reasoning), ("cache", cached)) if value is not None]
        detalhes = f" ({', '.join(extra)})" if extra else ""
        print(
            f"Chamada {n}: in={usage.get('input_tokens')} out={usage.get('output_tokens')}{detalhes}"
            f" | finish_reason={reply.response_metadata.get('finish_reason')}"
        )
        print(
            f"  latência no cliente={span.latency:.2f}s | servidor={server.get('total_time') or 0:.2f}s"
            f" | fila={server.get('queue_time') or 0:.2f}s | tentativas={span.attrs.get('attempts')}"
        )
        for retry in span.attrs.get("retries") or []:
            print(f"  retentado: {retry}")
    if res.error:
        print(f"Última chamada falhou: {res.error}")


mostrar_metricas(result)

### E3. Limites da conta agora

Faz **uma** chamada mínima e lê os cabeçalhos `x-ratelimit-*` da resposta: quantas requisições ainda restam no dia
(RPD) e quantos tokens restam neste minuto (TPM) para o modelo. Os limites valem para a organização inteira (todas as
chaves da sua conta).

Atenção: alguns modelos têm também um limite só de tokens de **saída** por minuto (OTPM; para o qwen no plano
gratuito testado em 2026-10-21, 1000). Ele **não** vem nesses cabeçalhos: aparece só no 429 (em E2, como
`RateLimitError OTPM`). Na prática, cabe mais ou menos um run por minuto.

In [ ]:
client = groq.Groq(api_key=GROQ_API_KEY, max_retries=0)
try:
    raw = client.chat.completions.with_raw_response.create(
        model=MODEL, messages=[{"role": "user", "content": "ok"}], max_completion_tokens=8
    )
    h = raw.headers
    print(f"Modelo: {MODEL}")
    print(
        f"Requisições restantes hoje: {h.get('x-ratelimit-remaining-requests')} de "
        f"{h.get('x-ratelimit-limit-requests')} (renova em {h.get('x-ratelimit-reset-requests')})"
    )
    print(
        f"Tokens restantes neste minuto: {h.get('x-ratelimit-remaining-tokens')} de "
        f"{h.get('x-ratelimit-limit-tokens')} (renova em {h.get('x-ratelimit-reset-tokens')})"
    )
except groq.RateLimitError as exc:
    print(f"429 agora: limite {rate_limit_window(exc)}; esperar {exc.response.headers.get('retry-after')} s.")
except groq.APIStatusError as exc:
    print(f"FALHA: {type(exc).__name__} ({exc.status_code})")

### E4. Testar outro modelo

Defina `OUTRO_MODELO` com um modelo da sua conta e rode: o agente é montado de novo com as mesmas peças e executa a
mesma `TASK`. Exemplo: `"openai/gpt-oss-120b"`, que falhou quando a resposta ia dentro do `finish` (400
`tool_use_failed`); com a resposta em texto, ainda não foi testado.
Depois, `mostrar_memoria(outro)` e `mostrar_metricas(outro)` mostram os detalhes. Cada execução gasta chamadas da cota.

In [ ]:
OUTRO_MODELO = None  # ex.: "openai/gpt-oss-120b"

if OUTRO_MODELO is None:
    print("Defina OUTRO_MODELO e rode de novo.")
else:
    outro_agente = Agent(
        llm=build_llm(OUTRO_MODELO, GROQ_API_KEY, temperature=config.temperature),
        registry=registry,
        memory=WorkingMemory(config.system_prompt),
        tracer=Tracer(),
        config=config,
    )
    outro = outro_agente.run(TASK)
    print()
    print(f"Modelo: {OUTRO_MODELO} | motivo: {outro.reason} | passos: {outro.steps} | erro: {outro.error}")
    print(f"Resposta final:\n{outro.answer}")
    print()
    print(format_report(outro.spans))